# 01 · ABTC Quickstart

**Adaptive Bayesian Thresholding Classifier**

A model-agnostic framework combining Bayesian posterior recalibration with optimal threshold selection.

---

## Overview

ABTC addresses the class imbalance problem through:
1. **Probability Calibration** - Isotonic regression for reliable posteriors
2. **Posterior Adjustment** - Bayesian recalibration with parameter α
3. **Optimal Threshold Selection** - Metric- or cost-driven optimization

---

## Search Architecture: Single-Layer Grid Search

### Path A (Metric-Driven)
- **Objective**: Maximize F1, F2, G-Mean, MCC, Balanced Accuracy, or PR-AUC
- **Method**: Joint grid search over (α, τ)
- **Grid**: 27α × 20τ = **540 evaluations**
- **Ranges**: α ∈ [0.1, 3.0], τ ∈ [0.01, 0.99]

### Path B (Cost-Sensitive)
- **Objective**: Minimize total misclassification cost
- **Method**: α search with analytical τ*
- **Formula**: τ* = C_FP / (C_FP + C_FN)
- **Grid**: 27α = **27 evaluations**

## Installation

In [ ]:
# Install ABTC package (run from repository root)
%pip install -e . -q

## Quick Start Example

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ABTC imports
from abtc import (
    isotonic_calibrator,
    alpha_adjust_posterior,
    joint_alpha_tau_search,
    alpha_search_analytical_tau,
    tau_cost,
    fbeta, gmean, mcc, balanced_acc
)

print("ABTC imported successfully!")

## Generate Imbalanced Dataset

In [ ]:
# Create imbalanced binary classification dataset
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    weights=[0.9, 0.1],  # 90% negative, 10% positive
    flip_y=0.01,
    random_state=42
)

# Split: 60% train, 20% validation, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

# Scale features
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f"Dataset Summary:")
print(f"  Train: {len(y_train)} samples ({np.mean(y_train)*100:.1f}% positive)")
print(f"  Val:   {len(y_val)} samples ({np.mean(y_val)*100:.1f}% positive)")
print(f"  Test:  {len(y_test)} samples ({np.mean(y_test)*100:.1f}% positive)")
print(f"  Imbalance Ratio: 1:{np.sum(y_train==0)/max(np.sum(y_train==1),1):.1f}")

## Step 1: Train Base Classifier

In [ ]:
# Train logistic regression as base classifier
base_model = LogisticRegression(solver='liblinear', max_iter=1000)
base_model.fit(X_train_s, y_train)

# Get raw probabilities
p_train_raw = base_model.predict_proba(X_train_s)[:, 1]
p_val_raw = base_model.predict_proba(X_val_s)[:, 1]
p_test_raw = base_model.predict_proba(X_test_s)[:, 1]

print("Base classifier trained.")
print(f"  Raw probability range: [{p_val_raw.min():.3f}, {p_val_raw.max():.3f}]")

## Step 2: Calibrate Probabilities

In [ ]:
# Apply isotonic calibration
calibrator, p_train_cal = isotonic_calibrator(y_train, p_train_raw)
p_val_cal = calibrator.transform(p_val_raw)
p_test_cal = calibrator.transform(p_test_raw)

# Calculate training prior
pi_train = float(np.mean(y_train))

print("Isotonic calibration applied.")
print(f"  Training prior π = {pi_train:.4f}")
print(f"  Calibrated probability range: [{p_val_cal.min():.3f}, {p_val_cal.max():.3f}]")

## Step 3A: Path A - Metric-Driven Optimization

Joint grid search over (α, τ) to maximize F2-score.

In [ ]:
# ABTC Search Configuration
N_ALPHA = 27  # α values in [0.1, 3.0]
N_TAU = 20    # τ values in [0.01, 0.99]
ALPHA_RANGE = (0.1, 3.0)
TAU_RANGE = (0.01, 0.99)

print("="*60)
print("ABTC SINGLE-LAYER SEARCH CONFIGURATION")
print("="*60)
print(f"α grid: {N_ALPHA} values in [{ALPHA_RANGE[0]}, {ALPHA_RANGE[1]}]")
print(f"τ grid: {N_TAU} values in [{TAU_RANGE[0]}, {TAU_RANGE[1]}]")
print(f"Path A evaluations: {N_ALPHA} × {N_TAU} = {N_ALPHA * N_TAU}")
print(f"Path B evaluations: {N_ALPHA} (τ analytical)")
print("="*60)

In [ ]:
# Path A: Optimize for F2-score
f2_metric = lambda yt, yp: fbeta(yt, yp, beta=2.0)

alpha_a, tau_a, score_a, n_eval_a = joint_alpha_tau_search(
    y_val=y_val,
    p_raw=p_val_cal,
    pi=pi_train,
    adjust_fn=alpha_adjust_posterior,
    metric_fn=f2_metric,
    n_alpha=N_ALPHA,
    n_tau=N_TAU,
    alpha_range=ALPHA_RANGE,
    tau_range=TAU_RANGE
)

print("\n" + "="*60)
print("PATH A RESULTS (Metric-Driven: F2-Score)")
print("="*60)
print(f"  Optimal α* = {alpha_a:.4f}")
print(f"  Optimal τ* = {tau_a:.4f}")
print(f"  Best F2    = {score_a:.4f}")
print(f"  Evaluations: {n_eval_a}")
print("="*60)

## Step 3B: Path B - Cost-Sensitive Optimization

α search with analytical τ* from cost ratio.

In [ ]:
# Cost configuration
COST_FP = 1.0   # Cost of False Positive
COST_FN = 10.0  # Cost of False Negative (10x more costly)

# Cost metric (negative because we minimize cost)
def cost_metric(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return COST_FP * fp + COST_FN * fn

# Path B: Cost-sensitive optimization
alpha_b, tau_b, score_b, n_eval_b = alpha_search_analytical_tau(
    y_val=y_val,
    p_raw=p_val_cal,
    pi=pi_train,
    adjust_fn=alpha_adjust_posterior,
    cost_fn=COST_FN,
    cost_fp=COST_FP,
    metric_fn=lambda yt, yp: -cost_metric(yt, yp),  # Negative for maximization
    n_alpha=N_ALPHA,
    alpha_range=ALPHA_RANGE
)

print("\n" + "="*60)
print("PATH B RESULTS (Cost-Sensitive)")
print("="*60)
print(f"  Cost Ratio: C_FN/C_FP = {COST_FN/COST_FP:.1f}")
print(f"  Analytical τ* = {tau_b:.4f}")
print(f"  Optimal α*    = {alpha_b:.4f}")
print(f"  Min Cost      = {-score_b:.1f}")
print(f"  Evaluations:  {n_eval_b}")
print("="*60)

## Step 4: Apply to Test Set

In [ ]:
# Apply Path A parameters to test set
p_test_adj_a = alpha_adjust_posterior(p_test_cal, alpha_a, pi_train)
y_pred_a = (p_test_adj_a >= tau_a).astype(int)

# Apply Path B parameters to test set
p_test_adj_b = alpha_adjust_posterior(p_test_cal, alpha_b, pi_train)
y_pred_b = (p_test_adj_b >= tau_b).astype(int)

# Baseline: default threshold 0.5
y_pred_baseline = (p_test_cal >= 0.5).astype(int)

print("Predictions generated for test set.")

## Step 5: Evaluate Results

In [ ]:
def evaluate_predictions(y_true, y_pred, p_score, name):
    """Compute all evaluation metrics."""
    from abtc import pr_auc
    return {
        'Method': name,
        'F1': fbeta(y_true, y_pred, 1.0),
        'F2': fbeta(y_true, y_pred, 2.0),
        'G-Mean': gmean(y_true, y_pred),
        'MCC': mcc(y_true, y_pred),
        'BalAcc': balanced_acc(y_true, y_pred),
        'PR-AUC': pr_auc(y_true, p_score),
        'Cost': cost_metric(y_true, y_pred)
    }

# Evaluate all methods
results = [
    evaluate_predictions(y_test, y_pred_baseline, p_test_cal, 'Baseline (τ=0.5)'),
    evaluate_predictions(y_test, y_pred_a, p_test_adj_a, f'ABTC Path A (α={alpha_a:.2f}, τ={tau_a:.2f})'),
    evaluate_predictions(y_test, y_pred_b, p_test_adj_b, f'ABTC Path B (α={alpha_b:.2f}, τ={tau_b:.2f})')
]

# Display results
df_results = pd.DataFrame(results)
print("\n" + "="*80)
print("TEST SET RESULTS")
print("="*80)
print(df_results.to_string(index=False))
print("="*80)

## Summary

### ABTC Single-Layer Search Architecture

| Path | Objective | Method | Evaluations |
|------|-----------|--------|-------------|
| **Path A** | Metric (F1, F2, etc.) | Joint (α, τ) grid | 27 × 20 = 540 |
| **Path B** | Cost | α search + analytical τ* | 27 |

### Key Formulas

**Posterior Adjustment:**
$$p_{adj} = \frac{p / \pi^\alpha}{p / \pi^\alpha + (1-p) / (1-\pi)^\alpha}$$

**Analytical Threshold (Path B):**
$$\tau^* = \frac{C_{FP}}{C_{FP} + C_{FN}}$$

**Decision Rule:**
$$\hat{y} = \mathbb{1}[p_{adj} \geq \tau^*]$$

In [ ]:
print("\n" + "="*60)
print("ABTC QUICKSTART COMPLETE")
print("="*60)
print(f"\nPath A: α*={alpha_a:.4f}, τ*={tau_a:.4f} ({n_eval_a} evals)")
print(f"Path B: α*={alpha_b:.4f}, τ*={tau_b:.4f} ({n_eval_b} evals)")
print("\nFor full reproduction, see: 02_Reproduce_Public_Results.ipynb")